In [2]:
import os
import pandas as pd
from hashlib import md5
from PIL import Image
import numpy as np

## Classes have been renamed, merged or newly created.

In [3]:
baseline_data_path = "/Users/lukasb/Documents/data/surfaceClassification/baseline"
noisy_data_path = "/Users/lukasb/Documents/data/surfaceClassification/noisy"

artifacts_path = "artifacts/noisy"

In [4]:
baseline_df_no_duplicates = pd.read_csv("artifacts/baseline/baseline_images_no_duplicates.csv")
print(len(baseline_df_no_duplicates))

5542


In [5]:
noisy_images = []

extensions = [".png", ".jpg", ".jpeg"]

for dirpath, dirnames, filenames in os.walk(noisy_data_path):
    for filename in filenames:
        if any(filename.endswith(ext) for ext in extensions):
            file_path = os.path.join(dirpath, filename)
            label = os.path.basename(dirpath)
            img = Image.open(file_path)
            file_path = file_path.removeprefix(noisy_data_path)
            image_id = md5(img.tobytes()).hexdigest()
            noisy_images.append({"image_id": image_id, "file_path": file_path, "label": label})

noisy_df = pd.DataFrame(noisy_images)
print(len(noisy_df))

5475


In [6]:
removed_df = baseline_df_no_duplicates[~baseline_df_no_duplicates["image_id"].isin(noisy_df["image_id"])]
len(removed_df)
removed_df.to_csv(os.path.join(artifacts_path, "removed_images.csv"), index=False)

In [10]:
duplicates_ids = noisy_df[noisy_df.duplicated(subset=['image_id'], keep="first")]['image_id'].unique()
df_no_duplicates = noisy_df.drop_duplicates(subset=["image_id"], keep="first").copy()
df_no_duplicates.to_csv(os.path.join(artifacts_path, "noisy_images_no_duplicates.csv"), index=False)
print(f"Rows without duplicates: {len(df_no_duplicates)}")

Rows without duplicates: 5475


In [8]:
removed = 0
for image_id in duplicates_ids:
    duplicate_rows = noisy_df[noisy_df["image_id"] == image_id]
    path = noisy_data_path + duplicate_rows.iloc[0]["file_path"]
    if os.path.exists(path):
        os.remove(path)
    removed += 1
print(f"Removed {removed} duplicate images.")

Removed 0 duplicate images.


## moved background images from "ok" to extra "background" class

In [9]:
noisy_images = []

extensions = [".png", ".jpg", ".jpeg"]

for dirpath, dirnames, filenames in os.walk(noisy_data_path):
    for filename in filenames:
        if any(filename.endswith(ext) for ext in extensions):
            file_path = os.path.join(dirpath, filename)
            label = os.path.basename(dirpath)
            img = Image.open(file_path)
            file_path = file_path.removeprefix(noisy_data_path)
            image_id = md5(img.tobytes()).hexdigest()
            noisy_images.append({"image_id": image_id, "file_path": file_path, "label": label})

noisy_df = pd.DataFrame(noisy_images)
print(len(noisy_df))

5475


## load similarities

In [1]:
best_treshold = float(0.92)

In [12]:
# Use the threshold found earlier (best_threshold is already defined in the notebook)
thr = best_treshold

similarity_path = os.path.join("artifacts/baseline", "similarity_pairs.csv")
similarity_df = pd.read_csv(similarity_path)

df_path = os.path.join(artifacts_path, "noisy_images_no_duplicates.csv")
df = pd.read_csv(df_path)

# Keep only edges with similarity >= threshold
edges_df = similarity_df[similarity_df["similarity"] >= thr][
    ["image_id_1", "image_id_2", "similarity"]
] .copy()

print(f"Threshold: {thr:.6f}")
print(f"Edges (similarity >= threshold): {len(edges_df):,}")

# --- Clique-style growth on undirected pairs ---
# A node can join a cluster only if it is similar to ALL members (sim >= thr).
# Pairs are treated as undirected, so we build a symmetric similarity lookup.
sim_map = {}
for r in edges_df.itertuples(index=False):
    a = str(r.image_id_1)
    b = str(r.image_id_2)
    if a == b:
        continue
    key = (a, b) if a <= b else (b, a)
    sim = float(r.similarity)
    if key not in sim_map or sim > sim_map[key]:
        sim_map[key] = sim

neighbors = {}
for (a, b), sim in sim_map.items():
    neighbors.setdefault(a, set()).add(b)
    neighbors.setdefault(b, set()).add(a)

nodes = sorted(neighbors.keys(), key=lambda n: len(neighbors[n]), reverse=True)
clusters = []
for node in nodes:
    placed = False
    for cluster in clusters:
        ok = True
        for member in cluster:
            key = (node, member) if node <= member else (member, node)
            sim = sim_map.get(key)
            if sim is None or sim < thr:
                ok = False
                break
        if ok:
            cluster.append(node)
            placed = True
            break
    if not placed:
        clusters.append([node])

clusters = [sorted(c) for c in clusters if len(c) >= 2]
print(f"Clique clusters (size >= 2): {len(clusters):,}")

# Build edge list from the clique clusters
edge_list = []
for cluster in clusters:
    for i in range(len(cluster)):
        for j in range(i + 1, len(cluster)):
            key = (cluster[i], cluster[j]) if cluster[i] <= cluster[j] else (cluster[j], cluster[i])
            sim = sim_map.get(key)
            if sim is not None and sim >= thr:
                edge_list.append((key[0], key[1], sim))
print(f"Clique edges: {len(edge_list):,}")

# --- Graph-based connected components ---
# Build adjacency list
graph = {}
for a, b, sim in edge_list:
    if a not in graph:
        graph[a] = set()
    if b not in graph:
        graph[b] = set()
    graph[a].add(b)
    graph[b].add(a)

# Find connected components via DFS/BFS
visited = set()
clusters = []
for node in graph:
    if node in visited:
        continue
    stack = [node]
    component = []
    visited.add(node)
    while stack:
        cur = stack.pop()
        component.append(cur)
        for nbr in graph[cur]:
            if nbr not in visited:
                visited.add(nbr)
                stack.append(nbr)
    if len(component) >= 2:
        clusters.append(sorted(component))

clusters.sort(key=len, reverse=True)

print(f"Clusters (size >= 2): {len(clusters):,}")
if clusters:
    print(f"Largest cluster size: {len(clusters[0])}")

# Create membership table
membership_rows = []
for cid, members in enumerate(clusters):
    for image_id in members:
        membership_rows.append({"cluster_id": cid, "image_id": image_id, "threshold": thr})

clusters_df = pd.DataFrame(membership_rows)

# Add manifest metadata if available
if not df.empty and not clusters_df.empty:
    clusters_df = clusters_df.merge(df, on="image_id", how="left")

# Save outputs
out_membership = os.path.join(artifacts_path, f"near_duplicate_clusters_thr_{thr:.3f}.csv")
clusters_df.to_csv(out_membership, index=False)
print(f"Saved cluster membership to: {out_membership}")

Threshold: 0.920000
Edges (similarity >= threshold): 64,525
Clique clusters (size >= 2): 1,392
Clique edges: 9,022
Clusters (size >= 2): 1,392
Largest cluster size: 14
Saved cluster membership to: artifacts/noisy/near_duplicate_clusters_thr_0.920.csv


In [17]:
import numpy as np

all_ids = df["image_id"].astype(str).unique().tolist()

clustered_ids = set(clusters_df["image_id"].astype(str).unique())
singleton_ids = [i for i in all_ids if i not in clustered_ids]
clustered_only_ids = [i for i in all_ids if i in clustered_ids]

n_total = len(all_ids)
if n_total == 0:
    raise ValueError("No images found for splitting.")

n_train = int(round(0.60 * n_total))
n_val = int(round(0.20 * n_total))
n_test = n_total - n_train - n_val

rng = np.random.default_rng(42)
rng.shuffle(singleton_ids)
rng.shuffle(clustered_only_ids)

val_ids = []
test_ids = []
train_ids = []

# Fill /test from singletons first
test_ids.extend(singleton_ids[:n_test])
remaining_singletons = singleton_ids[n_test:]

needed_val = n_val
available_for_val = remaining_singletons
if len(available_for_val) >= needed_val:
    val_ids.extend(available_for_val[:needed_val])
    remaining_singletons = available_for_val[needed_val:]
else:
    test_ids.extend(available_for_val)
    remaining_singletons = []

# Fill any remaining val/test slots from clustered images
remaining_val_slots = n_val - len(val_ids)
remaining_test_slots = n_test - len(test_ids)

if remaining_val_slots > 0:
    val_ids.extend(clustered_only_ids[:remaining_val_slots])
    clustered_only_ids = clustered_only_ids[remaining_val_slots:]

if remaining_test_slots > 0:
    test_ids.extend(clustered_only_ids[:remaining_test_slots])
    clustered_only_ids = clustered_only_ids[remaining_test_slots:]

# Put the rest into train
train_ids = clustered_only_ids + remaining_singletons

# Sanity checks
assert len(set(train_ids) & set(val_ids)) == 0
assert len(set(train_ids) & set(test_ids)) == 0
assert len(set(val_ids) & set(test_ids)) == 0
assert len(train_ids) + len(val_ids) + len(test_ids) == n_total

# Build split DataFrame
split_rows = []
for i in train_ids:
    split_rows.append({"image_id": i, "split": "train"})
for i in val_ids:
    split_rows.append({"image_id": i, "split": "val"})
for i in test_ids:
    split_rows.append({"image_id": i, "split": "test"})

split_df = pd.DataFrame(split_rows)

# Merge paths/labels for convenience
split_df = split_df.merge(df, on="image_id", how="left")

out_path = os.path.join(artifacts_path, "cluster_aware_split_60_20_20.csv")
split_df.to_csv(out_path, index=False)

# Report composition

def count_clustered(ids):
    return sum(1 for i in ids if i in clustered_ids)

print(f"Total images: {n_total}")
print(f"Train/Val/Test: {len(train_ids)}/{len(val_ids)}/{len(test_ids)}")
print(
    "Clustered in splits (train/val/test): "
    f"{count_clustered(train_ids)}/{count_clustered(val_ids)}/{count_clustered(test_ids)}"
)
print(
    "Singletons in splits (train/val/test): "
    f"{len(train_ids)-count_clustered(train_ids)}/"
    f"{len(val_ids)-count_clustered(val_ids)}/"
    f"{len(test_ids)-count_clustered(test_ids)}"
)
print(f"Saved split manifest to {out_path}")

Total images: 5475
Train/Val/Test: 3285/1095/1095
Clustered in splits (train/val/test): 3285/1095/585
Singletons in splits (train/val/test): 0/0/510
Saved split manifest to artifacts/noisy/cluster_aware_split_60_20_20.csv


In [18]:
from pathlib import Path
from shutil import copy2

split_df_path = os.path.join(artifacts_path, "cluster_aware_split_60_20_20.csv")
split_df = pd.read_csv(split_df_path)

out_root = Path("/Users/lukasb/Documents/data/surfaceClassification/noisy_clustered")

required_cols = {"file_path", "label", "split"}
missing_cols = required_cols - set(split_df.columns)
if missing_cols:
    raise ValueError(f"Split manifest missing columns: {missing_cols}")

# Create directories
for split in ["train", "val", "test"]:
    (out_root / split).mkdir(parents=True, exist_ok=True)

# Copy images into split/label folders
copied = 0
missing = 0
for row in split_df.itertuples(index=False):
    src = Path(noisy_data_path + row.file_path)
    if not src.exists():
        missing += 1
        continue
    dest_dir = out_root / row.split / row.label
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_path = dest_dir / src.name
    copy2(src, dest_path)
    copied += 1

print(f"Copied {copied} images to {out_root}")
if missing:
    print(f"Skipped {missing} missing files")

Copied 5475 images to /Users/lukasb/Documents/data/surfaceClassification/noisy_clustered


## Now all the test and val sets have been cleaned

In [20]:
noisy_images = []

extensions = [".png", ".jpg", ".jpeg"]

for dirpath, dirnames, filenames in os.walk(noisy_data_path):
    for filename in filenames:
        if any(filename.endswith(ext) for ext in extensions):
            file_path = os.path.join(dirpath, filename)
            label = os.path.basename(dirpath)
            img = Image.open(file_path)
            file_path = file_path.removeprefix(noisy_data_path)
            image_id = md5(img.tobytes()).hexdigest()
            noisy_images.append({"image_id": image_id, "file_path": file_path, "label": label})

noisy_df_cleaned = pd.DataFrame(noisy_images)
print(len(noisy_df_cleaned))
noisy_df_cleaned.to_csv(os.path.join(artifacts_path, "noisy_images_cleaned.csv"), index=False)

5475
